<a href="https://colab.research.google.com/github/fatemarampurawalaa-sudo/Lease_reader/blob/main/Lease_reader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q google-genai chromadb pypdf fpdf2

import time
import pandas as pd
import numpy as np
import chromadb
from pydantic import BaseModel
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

CHAT_MODEL = "gemini-flash-lite-latest"
EMBED_MODEL = "gemini-embedding-001"

print("Setup done ✅")

Setup done ✅


In [3]:
from fpdf import FPDF

lease_text = """RESIDENTIAL TENANCY CONTRACT

1. PARTIES
This contract is made between Mr. Ahmed Al Mansoori (the Landlord) and Ms. Priya Sharma (the Tenant).

2. PROPERTY
Apartment 1204, Marina Heights Tower, Dubai Marina, Dubai, UAE. The apartment is unfurnished and has 2 bedrooms.

3. TERM
The tenancy starts on 1 January 2027 and ends on 31 December 2027.

4. RENT
The annual rent is AED 95,000, payable in 4 cheques of AED 23,750 each, dated 1 January, 1 April, 1 July and 1 October 2027.

5. SECURITY DEPOSIT
The Tenant pays a security deposit of AED 4,750 at signing. It will be refunded within 30 days after the Tenant leaves, minus the cost of any damage.

6. PARKING AND FACILITIES
One covered parking space, number B2-118, is included. The Tenant may use the gym and swimming pool on the 3rd floor free of charge between 6 am and 11 pm.

7. UTILITIES
The Tenant pays DEWA electricity and water bills and internet. Chiller charges for air conditioning are included in the rent.

8. MAINTENANCE
Minor repairs under AED 1,000 are paid by the Tenant. Major repairs such as air conditioning, plumbing and electrical faults are paid by the Landlord.

9. PETS AND SUBLETTING
Pets are not allowed. Subletting is not allowed without written permission from the Landlord.

10. RENEWAL AND TERMINATION
If the Tenant wants to renew, they must inform the Landlord at least 90 days before the end date. If the Tenant leaves early, a penalty of two months' rent applies."""

pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=11)
pdf.multi_cell(0, 6, lease_text)
pdf.output("sample_lease.pdf")

print("PDF created ✅")

PDF created ✅


In [4]:
from pypdf import PdfReader

def read_pdf(path):
    reader = PdfReader(path)
    pages = [page.extract_text() for page in reader.pages]
    return "\n".join(pages)

text = read_pdf("sample_lease.pdf")

print("Pages:", len(PdfReader("sample_lease.pdf").pages))
print("Characters:", len(text))
print("---")
print(text[:500])

Pages: 1
Characters: 1441
---
RESIDENTIAL TENANCY CONTRACT
1. PARTIES
This contract is made between Mr. Ahmed Al Mansoori (the Landlord) and Ms. Priya Sharma (the Tenant).
2. PROPERTY
Apartment 1204, Marina Heights Tower, Dubai Marina, Dubai, UAE. The apartment is unfurnished and has 2
bedrooms.
3. TERM
The tenancy starts on 1 January 2027 and ends on 31 December 2027.
4. RENT
The annual rent is AED 95,000, payable in 4 cheques of AED 23,750 each, dated 1 January, 1 April, 1 July
and 1 October 2027.
5. SECURITY DEPOSIT
The T


In [6]:
import re

def chunk_by_section(text):
    parts = re.split(r"\n(?=\d+\.\s)", text)
    return [p.strip().replace("\n", " ") for p in parts if p.strip()]

chunks = chunk_by_section(text)

print("Number of chunks:", len(chunks))
for c in chunks:
    print("-", c[:70])

Number of chunks: 11
- RESIDENTIAL TENANCY CONTRACT
- 1. PARTIES This contract is made between Mr. Ahmed Al Mansoori (the La
- 2. PROPERTY Apartment 1204, Marina Heights Tower, Dubai Marina, Dubai,
- 3. TERM The tenancy starts on 1 January 2027 and ends on 31 December 2
- 4. RENT The annual rent is AED 95,000, payable in 4 cheques of AED 23,
- 5. SECURITY DEPOSIT The Tenant pays a security deposit of AED 4,750 at
- 6. PARKING AND FACILITIES One covered parking space, number B2-118, is
- 7. UTILITIES The Tenant pays DEWA electricity and water bills and inte
- 8. MAINTENANCE Minor repairs under AED 1,000 are paid by the Tenant. M
- 9. PETS AND SUBLETTING Pets are not allowed. Subletting is not allowed
- 10. RENEWAL AND TERMINATION If the Tenant wants to renew, they must in


In [8]:
def embed(text):
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return np.array(result.embeddings[0].values)

vectors = []
for c in chunks:
    vectors.append(embed(c))
    time.sleep(1)

db = chromadb.Client()
lease_db = db.get_or_create_collection(name="lease")
lease_db.add(
    ids=[f"section_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=[v.tolist() for v in vectors],
)

print("Chunks stored:", lease_db.count())

Chunks stored: 11


In [9]:
def ask_lease(question, top_k=2):
    # 1. Retrieve
    q_vector = embed(question)
    found = lease_db.query(query_embeddings=[q_vector.tolist()], n_results=top_k)
    ids = found["ids"][0]
    docs = found["documents"][0]
    context = "\n".join(f"[{i}] {d}" for i, d in zip(ids, docs))

    # 2. Augment
    prompt = f"""You answer questions about a tenancy contract.
Use ONLY the contract sections below.
If the answer is not in them, reply exactly: "I don't know, this is not in the contract."
Keep the answer to 1-2 sentences.
At the end write "Source:" and the [section id] you used, or "Source: none".

Contract sections:
{context}

Question: {question}"""

    # 3. Generate
    response = client.models.generate_content(model=CHAT_MODEL, contents=prompt)
    return response.text

questions = [
    "Is parking included, and which space?",
    "Who pays if the air conditioning breaks?",
    "Can I have a dog?",
    "Is there a concierge service?",
]

for q in questions:
    print("Q:", q)
    print("A:", ask_lease(q))
    print()
    time.sleep(2)

Q: Is parking included, and which space?
A: Yes, one covered parking space, number B2-118, is included.

Source: [section_6]

Q: Who pays if the air conditioning breaks?
A: Major repairs such as air conditioning are paid by the Landlord. 
Source: [section_8]

Q: Can I have a dog?
A: Pets are not allowed in the property. 

Source: [section_9]

Q: Is there a concierge service?
A: I don't know, this is not in the contract.
Source: none



In [11]:
class LeaseDetails(BaseModel):
    tenant_name: str
    landlord_name: str
    property_address: str
    bedrooms: int
    furnished: bool
    annual_rent: int
    number_of_cheques: int
    start_date: str
    end_date: str
    security_deposit: int | None = None
    parking_space: str | None = None
    pets_allowed: bool
    renewal_notice_days: int | None = None

def extract_lease(text):
    response = client.models.generate_content(
        model=CHAT_MODEL,
        contents=f"""Extract the lease details from this contract.
Dates must be in YYYY-MM-DD format. Amounts must be numbers in AED.
If a detail is not in the contract, leave it empty.

Contract:
{text}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=LeaseDetails,
        ),
    )
    return response.parsed.model_dump()

details = extract_lease(text)

lease_table = pd.DataFrame([details])
lease_table.to_csv("lease_details.csv", index=False)

lease_table.T

,0
tenant_name,Ms. Priya Sharma
landlord_name,Mr. Ahmed Al Mansoori
property_address,"Apartment 1204, Marina Heights Tower, Dubai Ma..."
bedrooms,2
furnished,False
annual_rent,95000
number_of_cheques,4
start_date,2027-01-01
end_date,2027-12-31
security_deposit,4750


In [13]:
tests = [
    {"question": "How much is the annual rent?", "must_contain": "95,000"},
    {"question": "When does the tenancy end?", "must_contain": "2027"},
    {"question": "Who pays the electricity bill?", "must_contain": "tenant"},
    {"question": "What is the penalty for leaving early?", "must_contain": "two months"},
    {"question": "Is there a rooftop garden?", "must_contain": "I don't know"},
]

results = []
for t in tests:
    answer = ask_lease(t["question"])
    ok = t["must_contain"].lower() in answer.lower()
    results.append({"question": t["question"], "expected": t["must_contain"], "correct": ok, "answer": answer})
    time.sleep(2)

qa_df = pd.DataFrame(results)
print("Q&A score:", qa_df["correct"].sum(), "out of", len(qa_df))
qa_df[["question", "expected", "correct"]]

Q&A score: 5 out of 5


,question,expected,correct
0,How much is the annual rent?,"95,000",True
1,When does the tenancy end?,2027,True
2,Who pays the electricity bill?,tenant,True
3,What is the penalty for leaving early?,two months,True
4,Is there a rooftop garden?,I don't know,True
